# Identificação de Duplicatas (E1)

Este notebook baixa o dataset de folhas de feijão do Kaggle e utiliza a biblioteca `ImageHash` para identificar imagens duplicadas ou muito similares, que poderiam enviesar o treinamento dos modelos.

In [5]:
import os
import pandas as pd
from PIL import Image
import imagehash
from tqdm.auto import tqdm


#!kaggle datasets download -d marquis03/bean-leaf-lesions-classification --unzip -p ./data

In [10]:
def extract_hashes(img_path):
    """
    Calcula os 4 tipos de hashes perceptuais para uma dada imagem.
    """
    try:
        img = Image.open(img_path)
        return {
            'path': img_path,
            'ahash': str(imagehash.average_hash(img)),
            'dhash': str(imagehash.dhash(img)),
            'phash': str(imagehash.phash(img)),
            'whash': str(imagehash.whash(img))
        }
    except Exception as e:
        print(f"Erro ao processar {img_path}: {e}")
        return None

# Percorrer a pasta de imagens
dataset_dir = './data' 
image_paths = []

if os.path.exists(dataset_dir):
    for root, _, files in os.walk(dataset_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(root, file))
                
print(f"Total de imagens encontradas: {len(image_paths)}")

Total de imagens encontradas: 1167


In [11]:
hashes_list = []
for path in tqdm(image_paths, desc="Calculando hashes"):
    h = extract_hashes(path)
    if h:
        hashes_list.append(h)

df_hashes = pd.DataFrame(hashes_list)
df_hashes.head()

Calculando hashes:   0%|          | 0/1167 [00:00<?, ?it/s]

,path,ahash,dhash,phash,whash
0,./data\train\angular_leaf_spot\angular_leaf_sp...,1cfffe7f0f030101,30f0b0dadcdfdb9b,91d46f3b194a741e,1cfefe7f0f030101
1,./data\train\angular_leaf_spot\angular_leaf_sp...,0108387e3e1e0c0e,5a7a72f8f83038dc,9de9333083ca4de6,0b183a7f7e1e0e0e
2,./data\train\angular_leaf_spot\angular_leaf_sp...,fcbc3c1ebff9c100,a160687868cb12e7,cfd668e800f103f7,f4bc3c1e3ff9c000
3,./data\train\angular_leaf_spot\angular_leaf_sp...,1d3c60f0f8fe7d18,f1ecc6020084e1d2,c12f0b9b79aa6147,1d2c60f0f8fe7d08
4,./data\train\angular_leaf_spot\angular_leaf_sp...,003038787d7c7c18,68e8f2d1d190d8f2,c12f7eb03d113cb8,2038387c7f7c7c38


In [ ]:
# Função auxiliar: analisa duplicatas exatas e near-duplicates para um único algoritmo de hash
HAMMING_THRESHOLD = 5 #valor padrão adotado academicamente, quanto menor mais semelhantes as iamgens terão de ser

def analyze_duplicates(df, hash_type, threshold=HAMMING_THRESHOLD):
    """
    Retorna (duplicatas_exatas, near_duplicates) considerando apenas o hash_type informado.
    """
    # Duplicatas exatas
    exact_duplicates = df[df.duplicated(subset=[hash_type], keep=False)].sort_values(by=hash_type)

    # Near-duplicates
    hashes = [imagehash.hex_to_hash(h) for h in df[hash_type]]
    paths = df['path'].tolist()

    near_duplicate_pairs = []
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            distance = hashes[i] - hashes[j]
            if distance <= threshold:
                near_duplicate_pairs.append({
                    'path_1': paths[i],
                    'path_2': paths[j],
                    'hamming_distance': distance
                })

    near_duplicates = pd.DataFrame(near_duplicate_pairs, columns=['path_1', 'path_2', 'hamming_distance'])
    if not near_duplicates.empty:
        near_duplicates = near_duplicates.sort_values(by='hamming_distance')

    return exact_duplicates, near_duplicates


In [13]:
# Análise usando ahash (average hash)
exact_ahash, near_ahash = analyze_duplicates(df_hashes, 'ahash')

print(f"[ahash] Duplicatas exatas: {len(exact_ahash)}")
display(exact_ahash.head(10))

print(f"[ahash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_ahash)}")
display(near_ahash.head(10))


[ahash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[ahash] Near-duplicates (distância <= 5): 24


,path_1,path_2,hamming_distance
1,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.137.jpg,3
4,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.12.jpg,3
6,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.319.jpg,3
2,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\angular_leaf_spot\angular_leaf_sp...,4
5,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.150.jpg,4
3,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.236.jpg,4
17,./data\train\healthy\healthy_train.128.jpg,./data\train\healthy\healthy_train.53.jpg,4
0,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\angular_leaf_spot\angular_leaf_sp...,5
8,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.27.jpg,5
9,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.311.jpg,5


In [14]:
# Análise usando dhash (difference hash)
exact_dhash, near_dhash = analyze_duplicates(df_hashes, 'dhash')

print(f"[dhash] Duplicatas exatas: {len(exact_dhash)}")
display(exact_dhash.head(10))

print(f"[dhash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_dhash)}")
display(near_dhash.head(10))


[dhash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[dhash] Near-duplicates (distância <= 5): 0


,path_1,path_2,hamming_distance


In [15]:
# Análise usando phash (perceptual hash)
exact_phash, near_phash = analyze_duplicates(df_hashes, 'phash')

print(f"[phash] Duplicatas exatas: {len(exact_phash)}")
display(exact_phash.head(10))

print(f"[phash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_phash)}")
display(near_phash.head(10))


[phash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[phash] Near-duplicates (distância <= 5): 0


,path_1,path_2,hamming_distance


In [16]:
# Análise usando whash (wavelet hash)
exact_whash, near_whash = analyze_duplicates(df_hashes, 'whash')

print(f"[whash] Duplicatas exatas: {len(exact_whash)}")
display(exact_whash.head(10))

print(f"[whash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_whash)}")
display(near_whash.head(10))


[whash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[whash] Near-duplicates (distância <= 5): 15


,path_1,path_2,hamming_distance
0,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.98.jpg,2
3,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.208.jpg,2
12,./data\train\healthy\healthy_train.130.jpg,./data\train\healthy\healthy_train.49.jpg,2
10,./data\train\healthy\healthy_train.102.jpg,./data\train\healthy\healthy_train.44.jpg,2
4,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.257.jpg,4
5,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.319.jpg,4
1,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\angular_leaf_spot\angular_leaf_sp...,4
2,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\bean_rust\bean_rust_train.46.jpg,4
7,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.49.jpg,4
6,./data\train\angular_leaf_spot\angular_leaf_sp...,./data\train\healthy\healthy_train.130.jpg,4


In [24]:
#  soma dos near-duplicates encontrados por cada algoritmo
near_duplicates_by_hash = {
    'ahash': len(near_ahash),
    'dhash': len(near_dhash),
    'phash': len(near_phash),
    'whash': len(near_whash),
}

total_near_duplicates = sum(near_duplicates_by_hash.values())
total_images = len(df_hashes)

df_resumo = pd.DataFrame([
    {'hash_type': hash_type, 'near_duplicates': count}
    for hash_type, count in near_duplicates_by_hash.items()
])
df_resumo.loc[len(df_resumo)] = ['TOTAL', total_near_duplicates]

print(f"Total de imagens no dataset: {total_images}")
print(f"Soma de near-duplicates encontrados (todos os algoritmos): {total_near_duplicates}")
print(f"{total_near_duplicates / total_images * 100:2f}% do total de imagens do dataset.")

display(df_resumo)


Total de imagens no dataset: 1167
Soma de near-duplicates encontrados (todos os algoritmos): 39
3.341902% do total de imagens do dataset.


,hash_type,near_duplicates
0,ahash,24
1,dhash,0
2,phash,0
3,whash,15
4,TOTAL,39


In [ ]:
# Quantas imagens precisam ser removidas para eliminar todos os pares near-duplicate?
#
# Cada par near-duplicate é uma "aresta" entre duas imagens. Se A é near-duplicate de B,
# e B é near-duplicate de C, remover só B já elimina os dois pares, então isso a resposta
# não é simplesmente 2x o número de pares, e sim o menor conjunto de imagens que "toca"
## todos os pares (problema clássico de cobertura mínima de vértices / minimum vertex cover).
from itertools import combinations

def build_edges():
    edges = set()
    for df in (near_ahash, near_dhash, near_phash, near_whash):
        for _, row in df.iterrows():
            edges.add(frozenset((row['path_1'], row['path_2']))) 
    return [tuple(e) for e in edges]

def connected_components(edges):
    parent = {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    for u, v in edges:
        parent.setdefault(u, u)
        parent.setdefault(v, v)
    for u, v in edges:
        union(u, v)

    components = {}
    for node in parent:
        components.setdefault(find(node), set()).add(node)
    return list(components.values())

def min_vertex_cover(nodes, edges):
    nodes = list(nodes)
    for k in range(len(nodes) + 1):
        for subset in combinations(nodes, k):
            subset_set = set(subset)
            if all(u in subset_set or v in subset_set for u, v in edges):
                return subset_set
    return set(nodes)

all_edges = build_edges()  # união de todos os pares, sem repetição, de todos os algoritmos
components = connected_components(all_edges)

images_to_remove = set()
for comp_nodes in components:
    comp_edges = [(u, v) for u, v in all_edges if u in comp_nodes and v in comp_nodes]
    images_to_remove |= min_vertex_cover(comp_nodes, comp_edges)

print(f"Pares near-duplicate únicos (união de ahash, dhash, phash e whash): {len(all_edges)}")
print(f"Grupos (componentes conectados) de imagens near-duplicate: {len(components)}")
print(f"Número mínimo de imagens a remover para zerar todos os near-duplicates: {len(images_to_remove)}")

display(pd.DataFrame(sorted(images_to_remove), columns=['imagem_a_remover']))


Pares near-duplicate únicos (união de ahash, dhash, phash e whash): 38
Grupos (componentes conectados) de imagens near-duplicate: 19
Número mínimo de imagens a remover para zerar todos os near-duplicates: 24


,imagem_a_remover
0,./data\train\angular_leaf_spot\angular_leaf_sp...
1,./data\train\angular_leaf_spot\angular_leaf_sp...
2,./data\train\angular_leaf_spot\angular_leaf_sp...
3,./data\train\angular_leaf_spot\angular_leaf_sp...
4,./data\train\angular_leaf_spot\angular_leaf_sp...
5,./data\train\bean_rust\bean_rust_train.120.jpg
6,./data\train\bean_rust\bean_rust_train.154.jpg
7,./data\train\bean_rust\bean_rust_train.236.jpg
8,./data\train\bean_rust\bean_rust_train.311.jpg
9,./data\train\bean_rust\bean_rust_train.323.jpg


In [25]:
# DataFrame final sem as imagens marcadas para remoção (near-duplicates)
df_sem_duplicatas = df_hashes[~df_hashes['path'].isin(images_to_remove)].reset_index(drop=True)

print(f"Total de imagens antes: {len(df_hashes)}")
print(f"Imagens removidas: {len(images_to_remove)}")
print(f"Total de imagens depois: {len(df_sem_duplicatas)}")

df_sem_duplicatas.head()


Total de imagens antes: 1167
Imagens removidas: 24
Total de imagens depois: 1143


,path,ahash,dhash,phash,whash
0,./data\train\angular_leaf_spot\angular_leaf_sp...,1cfffe7f0f030101,30f0b0dadcdfdb9b,91d46f3b194a741e,1cfefe7f0f030101
1,./data\train\angular_leaf_spot\angular_leaf_sp...,0108387e3e1e0c0e,5a7a72f8f83038dc,9de9333083ca4de6,0b183a7f7e1e0e0e
2,./data\train\angular_leaf_spot\angular_leaf_sp...,fcbc3c1ebff9c100,a160687868cb12e7,cfd668e800f103f7,f4bc3c1e3ff9c000
3,./data\train\angular_leaf_spot\angular_leaf_sp...,1d3c60f0f8fe7d18,f1ecc6020084e1d2,c12f0b9b79aa6147,1d2c60f0f8fe7d08
4,./data\train\angular_leaf_spot\angular_leaf_sp...,003038787d7c7c18,68e8f2d1d190d8f2,c12f7eb03d113cb8,2038387c7f7c7c38


In [26]:
# Reanalisando ahash e whash no df_sem_duplicatas, para conferir se os near-duplicates foram eliminados
exact_ahash_clean, near_ahash_clean = analyze_duplicates(df_sem_duplicatas, 'ahash')
exact_whash_clean, near_whash_clean = analyze_duplicates(df_sem_duplicatas, 'whash')

print(f"[ahash] Duplicatas exatas: {len(exact_ahash_clean)}")
display(exact_ahash_clean.head(10))

print(f"[ahash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_ahash_clean)}")
display(near_ahash_clean.head(10))

print(f"[whash] Duplicatas exatas: {len(exact_whash_clean)}")
display(exact_whash_clean.head(10))

print(f"[whash] Near-duplicates (distância <= {HAMMING_THRESHOLD}): {len(near_whash_clean)}")
display(near_whash_clean.head(10))


[ahash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[ahash] Near-duplicates (distância <= 5): 0


,path_1,path_2,hamming_distance


[whash] Duplicatas exatas: 0


,path,ahash,dhash,phash,whash


[whash] Near-duplicates (distância <= 5): 0


,path_1,path_2,hamming_distance
